In [ ]:
import csv
import json
import hashlib
import requests
import datetime
#from humanfriendly import format_timespan
import pandas as pd
#from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
import json
#from pymongo import MongoClient
#import pymysql
from datetime import datetime,timedelta
#import matplotlib.pyplot as plt
import ast
import psycopg2
import tqdm
from pandas import json_normalize
import urllib
import cv2

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 100000)



In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py



In [ ]:
def fetch_aadhaar_card_doc_store(user_code):
    url = DOC_STORE_API_BASE+"/documents/byentityids"
    payload="{\n    \"entityType\": \"OPERATOR\",\n    \"entityIds\": [ \""+ str(user_code) +"\"] ,\n    \"documentTypes\": [\n        \"AADHAAR_CARD\"    ]\n}"
    headers = {
          'Content-Type': 'application/json'
        } 
    response = requests.request("POST", url, headers=headers, data = payload)
    json_dat = json.loads(response.text)
    return json_dat

In [ ]:
dowload_set=pd.read_csv('OCR_batch_24_to_be_sent.csv').rename(columns={'operator_code':'user_code'})
dowload_set=dowload_set.dropna(how='all', axis=1)
dowload_set=dowload_set.dropna(how='all', axis=0)
#dowload_set.head(2)

In [ ]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation) for im in im_list]
    return cv2.hconcat(im_list_resize)

In [ ]:
def download_aadhaar (user_code):
        print(user_code)
    #try : 
        data=dowload_set[dowload_set['user_code']==user_code]
        data=data.reset_index()
        json_dat = fetch_aadhaar_card_doc_store(data['user_code'][0])
        
        if len(json_dat['data']['documents']) >0 :
            norm_json = json_normalize(json_dat['data']['documents'])
            
            
            adhaar_frnt = norm_json['frontImgUrl'][0]
            adhaar_frnt_name =  str(data['bank_user_id'][0]) + '_adhaar_frnt.jpg'
            adhaar_frnt_path = 'data/{}'.format(adhaar_frnt_name)
            
            adhaar_back = norm_json['backImgUrl'][0]
            adhaar_back_name =  str(data['bank_user_id'][0]) + '_adhaar_back.jpg'
            adhaar_back_path='data/{}'.format(adhaar_back_name)
            
            urllib.request.urlretrieve(adhaar_frnt, adhaar_frnt_path)
            urllib.request.urlretrieve(adhaar_back, adhaar_back_path)
            
            
            img1 = cv2.imread(adhaar_frnt_path)
            img2 = cv2.imread(adhaar_back_path)
            
            try :
                im_h_resize = hconcat_resize_min([img1, img2])
                concat_image_name= str(data['bank_user_id'][0]) + '_adhaar_concat.jpg'

                cv2.imwrite('data/{}'.format(concat_image_name), im_h_resize)


                return 'doc_downloaded_for_{}'.format(user_code)
            except : 
                print('probably doc dont exist for '+user_code)

        else :
            return 'doc_not_downloaded_for_{}_no_doc_in_doc_store'.format(user_code)
    #except as e: 
        #return e+'doc_not_downloaded_for_{}_error'.format(user_code)

In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
pool = ThreadPool(10)
reuslts=pool.map(download_aadhaar,dowload_set['user_code'])
#data=pd.DataFrame(reuslts)
pool.close()
pool.join()